In [1]:
import numpy as np
import tifffile as tiff
from skimage.transform import resize
from tensorflow.keras.models import load_model

IMG_SIZE = (64, 64)

RED_BAND = 3
NIR_BAND = 7
EPSILON = 1e-6

def preprocess_image(image_path):
    # Read original 13-band image
    img = tiff.imread(image_path)

    if img.shape[-1] != 13:
        raise ValueError(f"Expected 13 bands, got {img.shape[-1]}")

    # Resize image
    img = resize(
        img,
        (IMG_SIZE[0], IMG_SIZE[1], 13),
        preserve_range=True,
        anti_aliasing=True
    ).astype(np.float32)

    # Compute NDVI
    red = img[:, :, RED_BAND]
    nir = img[:, :, NIR_BAND]

    ndvi = (nir - red) / (nir + red + EPSILON)

    # Add ONLY NDVI as the 14th channel
    img = np.concatenate(
        [img, ndvi[..., np.newaxis]],
        axis=-1
    )

    # Check final shape
    print("Input shape:", img.shape)   # Should be (64, 64, 14)

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    return img

# Load model
model = load_model("SpectrumNet_final.h5")

print("Model expects:", model.input_shape)

# Image path
image_path = r"C:\college\CourseDataSci\final 2026\EuroSAT\allBands\Residential\Residential_58.tif"

# Preprocess
img = preprocess_image(image_path)

# Predict
prediction = model.predict(img, verbose=0)

predicted_class = np.argmax(prediction)

CLASSES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake"
]

print("Predicted:", CLASSES[predicted_class])
print("Confidence:", float(prediction[0][predicted_class]))

Model expects: (None, 64, 64, 14)
Input shape: (64, 64, 14)
Predicted: Residential
Confidence: 0.9974507689476013
